In [2]:
import pandas as pd
import numpy as np

Chargement des données

MOVIES : liste des films et leur genre

In [19]:
movies = pd.read_csv('movies.csv')

# Remplacement du genre (no genres listed) par NaN
movies['genres'] = movies['genres'].replace('(no genres listed)', np.nan)

# Séparation des genres en listes
movies = movies.join(
    movies['genres'].str.get_dummies(sep='|').add_prefix('genre_')
)

movies = movies.drop(columns=['genres'])

print(movies.shape)

movies.head()

(27278, 21)


,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [20]:
# titres de film en doublon
dup_titles = movies[movies.duplicated(subset='title', keep=False)].sort_values('title')
#affichons les films en doublon avec tous leurs genres et leurs movieId dans l'ordre croissant
#dup_titles[['movieId', 'title']]
pd.set_option('display.max_columns', None)
print(dup_titles.sort_values(['title', 'movieId']).to_string())


       movieId                                title  genre_Action  genre_Adventure  genre_Animation  genre_Children  genre_Comedy  genre_Crime  genre_Documentary  genre_Drama  genre_Fantasy  genre_Film-Noir  genre_Horror  genre_IMAX  genre_Musical  genre_Mystery  genre_Romance  genre_Sci-Fi  genre_Thriller  genre_War  genre_Western
20923   102190  20,000 Leagues Under the Sea (1997)             0                1                0               0             0            0                  0            0              0                0             0           0              0              0              1             1               0          0              0
24064   114130  20,000 Leagues Under the Sea (1997)             0                0                0               0             0            0                  0            0              0                0             0           0              0              0              1             1               0          0              

On a 16 films en doublon et on peut voir qu'ils n'ont pas forcément le même genre. On va regrouper garder la première occurence et la compléter éventuellement avec le genre du second

In [22]:
#on va garder le premier film pour chaque titre (id le plus petit) et compléter avec les genres des autres films en doublon
genre_cols = [col for col in dup_titles.columns if col.startswith('genre_')]

# dup_titles a une ligne par film en doublon (movieId est un simple entier, pas une liste) :
# on regroupe donc directement par titre, en gardant le plus petit movieId
# et en fusionnant les genres avec un OR (max sur des 0/1 = any())
merged = dup_titles.groupby('title', as_index=False).agg(
    movieId=('movieId', 'min'),
    **{col: (col, 'max') for col in genre_cols}
)

# on retire de movies toutes les lignes correspondant aux films en doublon...
movies = movies[~movies['movieId'].isin(dup_titles['movieId'])]
# ...et on rajoute les versions fusionnées (une seule ligne par titre en doublon)
movies = pd.concat([movies, merged], ignore_index=True)

movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 27262 entries, 0 to 27261
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   movieId            27262 non-null  int64
 1   title              27262 non-null  str  
 2   genre_Action       27262 non-null  int64
 3   genre_Adventure    27262 non-null  int64
 4   genre_Animation    27262 non-null  int64
 5   genre_Children     27262 non-null  int64
 6   genre_Comedy       27262 non-null  int64
 7   genre_Crime        27262 non-null  int64
 8   genre_Documentary  27262 non-null  int64
 9   genre_Drama        27262 non-null  int64
 10  genre_Fantasy      27262 non-null  int64
 11  genre_Film-Noir    27262 non-null  int64
 12  genre_Horror       27262 non-null  int64
 13  genre_IMAX         27262 non-null  int64
 14  genre_Musical      27262 non-null  int64
 15  genre_Mystery      27262 non-null  int64
 16  genre_Romance      27262 non-null  int64
 17  genre_Sci-Fi       2726

In [23]:
#afficher les lignes de movies dont le movieId est dans dup_titles
movies[movies['movieId'].isin(dup_titles['movieId'])]

,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
27246,102190,"20,000 Leagues Under the Sea (1997)",0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0
27247,588,Aladdin (1992),0,1,1,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0
27248,104035,Beneath (2013),0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
27249,66140,Blackout (2007),0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0
27250,42015,Casanova (2005),1,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
27251,47254,Chaos (2005),1,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0
27252,104155,Clear History (2013),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
27253,93279,Darling (2007),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
27254,838,Emma (1996),0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
27255,97773,"Girl, The (2012)",0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


Les doublons sont supprimés et les genres fusionnés

RATINGS : notes des films par les utilisateurs

In [48]:
ratings = pd.read_csv('ratings.csv')

ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')

display(ratings.head())
ratings.info(show_counts=True)

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


<class 'pandas.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 4 columns):
 #   Column     Non-Null Count     Dtype        
---  ------     --------------     -----        
 0   userId     20000263 non-null  int64        
 1   movieId    20000263 non-null  int64        
 2   rating     20000263 non-null  float64      
 3   timestamp  20000263 non-null  datetime64[s]
dtypes: datetime64[s](1), float64(1), int64(2)
memory usage: 610.4 MB


In [49]:
# on va remplacer dans ratings les movieId dédoublonnés
mapping = {
    114130: 102190,
    114240: 588,
    115777: 104035,
    85070: 66140,
    128862: 42015,
    67459: 47254,
    122940: 104155,
    130062: 93279,
    26958: 838,
    101212: 97773,
    65665: 3598,
    128991: 111519,
    26982: 1788,
    80330: 48682,
    121586: 113459,
    64997: 34048
}

ratings['movieId'] = ratings['movieId'].replace(mapping)



In [52]:
# vérifions dans ratings si on a des doublons de (userId, movieId)
dup_ratings = ratings[ratings.duplicated(subset=['userId', 'movieId'], keep=False)].sort_values(['userId', 'movieId'])
display(dup_ratings.shape)
print(dup_ratings.to_string())

(648, 4)

          userId  movieId  rating           timestamp
1334          11    34048     5.0 2009-01-01 05:41:29
1444          11    34048     5.0 2009-08-25 03:20:46
79227        572    34048     4.0 2011-12-19 04:51:44
79521        572    34048     4.0 2011-12-20 02:56:13
82107        586    34048     4.0 2007-04-10 05:44:03
82402        586    34048     3.0 2009-01-04 10:25:21
295700      2024    34048     4.0 2006-01-08 16:04:09
295864      2024    34048     3.5 2010-07-24 12:41:24
321203      2194    34048     4.5 2009-06-08 19:06:18
321212      2194    34048     1.0 2009-06-08 19:06:10
402308      2742    34048     2.0 2012-12-26 13:12:56
402410      2742    34048     2.0 2012-12-26 13:17:13
527364      3571    34048     2.5 2010-11-19 23:37:38
527577      3571    34048     1.5 2010-11-20 17:30:58
710980      4734    34048     4.5 2009-02-01 21:25:03
711045      4734    34048     4.5 2009-02-01 21:25:05
750355      4999    34048     2.0 2009-02-08 21:11:22
750396      4999    34048   

In [43]:
# il y a des doublons, on va garder la ligne avec le timestamp le plus récent (max)
ratings = ratings.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId'], keep='last')

In [46]:
ratings.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 19999939 entries, 4182421 to 12675921
Data columns (total 4 columns):
 #   Column     Non-Null Count     Dtype        
---  ------     --------------     -----        
 0   userId     19999939 non-null  int64        
 1   movieId    19999939 non-null  int64        
 2   rating     19999939 non-null  float64      
 3   timestamp  19999939 non-null  datetime64[s]
dtypes: datetime64[s](1), float64(1), int64(2)
memory usage: 762.9 MB


In [84]:
# on rajoute dans ratings les informations de film (title et genres) depuis movies
ratings = ratings.merge(movies[['movieId', 'title'] + genre_cols], on='movieId', how='left')
display(ratings.head())

,userId,movieId,rating,timestamp,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,1,0,0,0
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0


TAGS : tag des films par les utilisateurs

In [62]:
tags = pd.read_csv('tags.csv')

tags = tags.dropna()
tags['timestamp'] = pd.to_datetime(tags['timestamp'], unit='s')

display(tags.head())
tags.info()

,userId,movieId,tag,timestamp
0,18,4141,Mark Waters,2009-04-24 18:19:40
1,65,208,dark hero,2013-05-10 01:41:18
2,65,353,dark hero,2013-05-10 01:41:19
3,65,521,noir thriller,2013-05-10 01:39:43
4,65,592,dark hero,2013-05-10 01:41:18


<class 'pandas.DataFrame'>
Index: 465548 entries, 0 to 465563
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype        
---  ------     --------------   -----        
 0   userId     465548 non-null  int64        
 1   movieId    465548 non-null  int64        
 2   tag        465548 non-null  str          
 3   timestamp  465548 non-null  datetime64[s]
dtypes: datetime64[s](1), int64(2), str(1)
memory usage: 17.8 MB


In [63]:
# on va remplacer dans tags les movieId dédoublonnés
tags['movieId'] = tags['movieId'].replace(mapping)

In [64]:
# vérifions dans tags si on a des doublons de (userId, movieId, tag)
dup_tags = tags[tags.duplicated(subset=['userId', 'movieId', 'tag'], keep=False)].sort_values(['userId', 'movieId', 'tag'])
display(dup_tags.shape)
print(dup_tags.to_string())

(26, 4)

        userId  movieId                                          tag           timestamp
4999      1741      838                            adapted from:book 2007-08-12 12:58:47
8695      1741      838                            adapted from:book 2013-05-14 23:57:42
5000      1741      838                           author:Jane Austen 2007-08-12 12:58:47
8696      1741      838                           author:Jane Austen 2013-05-14 23:57:42
82251    20845     3598                                       Hamlet 2009-12-15 00:38:41
83065    20845     3598                                       Hamlet 2009-12-15 00:38:48
294042   88738     1788  easily confused with other movie(s) (title) 2013-12-17 11:04:10
301405   88738     1788  easily confused with other movie(s) (title) 2013-12-17 11:05:12
296424   88738     3598                                  Shakespeare 2009-07-30 20:04:44
305597   88738     3598                                  Shakespeare 2013-12-17 11:02:40
296422   88738     35

In [65]:
# il y a des doublons, on va garder la ligne avec le timestamp le plus récent (max)
tags = tags.sort_values('timestamp').drop_duplicates(subset=['userId', 'movieId', 'tag'], keep='last')

In [66]:
# vérifions dans tags si on a des doublons de (userId, movieId, tag)
dup_tags = tags[tags.duplicated(subset=['userId', 'movieId', 'tag'], keep=False)].sort_values(['userId', 'movieId', 'tag'])
display(dup_tags.shape)
print(dup_tags.to_string())

(0, 4)

Empty DataFrame
Columns: [userId, movieId, tag, timestamp]
Index: []


In [78]:
print(f"tags: {tags.shape}")   

tags: (465535, 4)


In [80]:
# dans le dataframe tags nous avons plusieurs lignes pour un film, soit une ligne par tag. On va regrouper les tags en une seule ligne par film
tags = (
    tags
    .groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)
display(tags.head(2))
tags.shape

,movieId,tag
0,1,the boys classic pixar Disney pixar cgi animat...
1,2,monkey For children game animals kid flick Chi...


(19536, 2)

In [81]:
# on rajoute dans tags les informations de film (title et genres) depuis movies
tags = tags.merge(movies[['movieId', 'title'] + genre_cols], on='movieId', how='left')
display(tags.head())

,movieId,tag,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,the boys classic pixar Disney pixar cgi animat...,Toy Story (1995),0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,monkey For children game animals kid flick Chi...,Jumanji (1995),0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,Funniest Movies sequel fever comedinha de velh...,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,chick flick revenge characters characters chic...,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,wedding sequel family pregnancy remake sequel ...,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


SCORE MOVIE : GENOME SCORES & GENOME TAGS

In [123]:
genome_scores = pd.read_csv('genome-scores.csv')
display(genome_scores.head())
genome_scores.info(show_counts=True)

,movieId,tagId,relevance
0,1,1,0.02500
1,1,2,0.02500
2,1,3,0.05775
3,1,4,0.09675
4,1,5,0.14675


<class 'pandas.DataFrame'>
RangeIndex: 11709768 entries, 0 to 11709767
Data columns (total 3 columns):
 #   Column     Non-Null Count     Dtype  
---  ------     --------------     -----  
 0   movieId    11709768 non-null  int64  
 1   tagId      11709768 non-null  int64  
 2   relevance  11709768 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 268.0 MB


In [124]:
# On définit un seuil de pertinence pour les tags. on ne garde que les tags vraiment représentatifs du film
# c'est à dire avec un score supérieur au seuil de pertinence
SEUIL_PERTINENCE = 0.5
genome_scores = genome_scores[genome_scores['relevance'] > SEUIL_PERTINENCE]
genome_scores.shape

(459572, 3)

In [125]:
# vérifions dans genome_scores si on a des doublons de (movieId, tagId)
dup_genome_scores = genome_scores[genome_scores.duplicated(subset=['movieId', 'tagId'], keep=False)].sort_values(['movieId', 'tagId'])
display(dup_genome_scores.shape)
print(dup_genome_scores.to_string())

(0, 3)

Empty DataFrame
Columns: [movieId, tagId, relevance]
Index: []


In [126]:
# pas de doublon, on va remplacer dans genome_scores les movieId dédoublonnés
genome_scores['movieId'] = genome_scores['movieId'].replace(mapping)

In [127]:
# vérifions dans genome_scores si on a des doublons de (movieId, tagId)
dup_genome_scores = genome_scores[genome_scores.duplicated(subset=['movieId', 'tagId'], keep=False)].sort_values(['movieId', 'tagId'])
display(dup_genome_scores.shape)
print(dup_genome_scores.to_string())

(84, 3)

         movieId  tagId  relevance
8407002    34048     19    0.88025
9696306    34048     19    0.54025
8407005    34048     22    0.83450
9696309    34048     22    0.65250
8407026    34048     43    0.99200
9696330    34048     43    0.98250
8407027    34048     44    0.98950
9696331    34048     44    0.95700
8407028    34048     45    0.98275
9696332    34048     45    0.97675
8407083    34048    100    0.51875
9696387    34048    100    0.60125
8407090    34048    107    0.78225
9696394    34048    107    0.94925
8407109    34048    126    0.64225
9696413    34048    126    0.51075
8407112    34048    129    0.61300
9696416    34048    129    0.52700
8407115    34048    132    0.92250
9696419    34048    132    0.93300
8407171    34048    188    0.79350
9696475    34048    188    0.75450
8407176    34048    193    0.61200
9696480    34048    193    0.51125
8407285    34048    302    0.60550
9696589    34048    302    0.71925
8407291    34048    308    0.56575
9696595    34048    

In [128]:
# on va dédoublonner genome_scores en gardant la ligne avec le score le plus élevé (max)
genome_scores = genome_scores.sort_values('relevance').drop_duplicates(subset=['movieId', 'tagId'], keep='last')

In [129]:
# vérifions dans genome_scores si on a des doublons de (movieId, tagId)
dup_genome_scores = genome_scores[genome_scores.duplicated(subset=['movieId', 'tagId'], keep=False)].sort_values(['movieId', 'tagId'])
display(dup_genome_scores.shape)
print(dup_genome_scores.to_string())

(0, 3)

Empty DataFrame
Columns: [movieId, tagId, relevance]
Index: []


In [130]:
genome_tags = pd.read_csv('genome-tags.csv')
display(genome_tags.head())
genome_tags.info()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


<class 'pandas.DataFrame'>
RangeIndex: 1128 entries, 0 to 1127
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   tagId   1128 non-null   int64
 1   tag     1128 non-null   str  
dtypes: int64(1), str(1)
memory usage: 17.8 KB


In [131]:
# Fusion genome_scores + genome_tags pour obtenir les noms des tags
genome_scores = genome_scores.merge(genome_tags, on='tagId', how='left')
display(genome_scores.head())

,movieId,tagId,relevance,tag
0,6047,356,0.50025,entirely dialogue
1,99917,242,0.50025,complicated
2,48342,551,0.50025,intelligent
3,55995,288,0.50025,dark hero
4,5951,188,0.50025,catastrophe


In [132]:
# Pour chaque film : concaténation des tags
genome_scores = (
    genome_scores
    .groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)

In [133]:
# on rajoute dans genome_scores les informations de film (title uniquement) depuis movies
genome_scores = genome_scores.merge(movies[['movieId', 'title']], on='movieId', how='left')
display(genome_scores.head())

,movieId,tag,title
0,1,talking animals destiny sentimental fairy tale...,Toy Story (1995)
1,2,happy ending predictable runaway good action d...,Jumanji (1995)
2,3,silly family pg-13 chase good mentor fish roma...,Grumpier Old Men (1995)
3,4,love story friendship divorce feel good movie ...,Waiting to Exhale (1995)
4,5,goofy culture clash cute! mentor crappy sequel...,Father of the Bride Part II (1995)


In [134]:
# dimensions finales des DataFrames
print(f"ratings: {ratings.shape}")
print(f"tags: {tags.shape}")   
print(f"genome_scores: {genome_scores.shape}")     

ratings: (20000263, 24)
tags: (19536, 22)
genome_scores: (10380, 3)


In [135]:
# sauvegarde des DataFrames prétraités dans des fichiers CSV
ratings.to_csv('ratings_preprocessed.csv', index=False)
tags.to_csv('tags_preprocessed.csv', index=False)   
genome_scores.to_csv('genome_scores_preprocessed.csv', index=False)